In [1]:
!pip install xarray netCDF4 pandas numpy pyarrow openpyxl tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.3 MB/s eta 0:00:00


In [2]:
import re
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from datetime import datetime, timedelta
from tqdm import tqdm

# ============================================================
# NASA MODIS-Aqua Level-3 8-day CHL via OPeNDAP
# Product: AQUA_MODIS L3m 8D CHL chlor_a 4km
# Variable: chlor_a
# ============================================================

OUTPUT_DIR = Path("modis_chlorophyll_data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2020
END_YEAR = 2024

BASINS = {
    "Bay_of_Bengal": {
        "lat_min": 5.0,
        "lat_max": 22.0,
        "lon_min": 80.0,
        "lon_max": 98.0
    },
    "Arabian_Sea": {
        "lat_min": 5.0,
        "lat_max": 22.0,
        "lon_min": 60.0,
        "lon_max": 75.0
    }
}


def generate_8day_periods(year):
    start = datetime(year, 1, 1)
    end_year = datetime(year, 12, 31)

    periods = []
    current = start

    while current <= end_year:
        period_start = current
        period_end = min(current + timedelta(days=7), end_year)

        periods.append((period_start, period_end))

        current = period_end + timedelta(days=1)

    return periods


def build_opendap_url(period_start, period_end):
    year = period_start.strftime("%Y")
    mmdd = period_start.strftime("%m%d")

    start_str = period_start.strftime("%Y%m%d")
    end_str = period_end.strftime("%Y%m%d")

    filename = f"AQUA_MODIS.{start_str}_{end_str}.L3m.8D.CHL.chlor_a.4km.nc"

    url = (
        f"https://oceandata.sci.gsfc.nasa.gov/opendap/MODISA/L3SMI/"
        f"{year}/{mmdd}/{filename}"
    )

    return url, filename


def to_025_degree_grid(values):
    return np.floor(values / 0.25) * 0.25 + 0.125


def select_latlon_region(ds, lat_min, lat_max, lon_min, lon_max):
    # MODIS lat may be ascending or descending; handle both safely.
    lat_values = ds["lat"].values
    lon_values = ds["lon"].values

    if lat_values[0] > lat_values[-1]:
        lat_slice = slice(lat_max, lat_min)
    else:
        lat_slice = slice(lat_min, lat_max)

    if lon_values[0] > lon_values[-1]:
        lon_slice = slice(lon_max, lon_min)
    else:
        lon_slice = slice(lon_min, lon_max)

    return ds.sel(lat=lat_slice, lon=lon_slice)


def process_one_opendap_file(period_start, period_end):
    url, filename = build_opendap_url(period_start, period_end)

    print("\nProcessing:", filename)

    date_start = pd.to_datetime(period_start)
    date_end = pd.to_datetime(period_end)
    date_center = date_start + (date_end - date_start) / 2

    frames = []

    try:
        ds = xr.open_dataset(url, engine="netcdf4")

        if "chlor_a" not in ds.data_vars:
            print("chlor_a not found. Available variables:", list(ds.data_vars))
            ds.close()
            return pd.DataFrame()

        for basin_name, b in BASINS.items():
            try:
                sub = select_latlon_region(
                    ds,
                    b["lat_min"],
                    b["lat_max"],
                    b["lon_min"],
                    b["lon_max"]
                )

                # Load only the small regional subset
                chl = sub["chlor_a"].load()

                df = chl.to_dataframe().reset_index()

                df = df.rename(columns={
                    "lat": "latitude",
                    "lon": "longitude"
                })

                df["chlor_a"] = pd.to_numeric(df["chlor_a"], errors="coerce")

                # Clean invalid values
                df = df.replace([np.inf, -np.inf], np.nan)
                df = df.dropna(subset=["chlor_a"])
                df = df[df["chlor_a"] > 0].copy()

                if df.empty:
                    print("No valid chlorophyll for:", basin_name)
                    continue

                df["basin"] = basin_name
                df["date_start"] = date_start
                df["date_end"] = date_end
                df["date_center"] = date_center

                df["lat_025"] = to_025_degree_grid(df["latitude"])
                df["lon_025"] = to_025_degree_grid(df["longitude"])

                df_025 = (
                    df.groupby(
                        ["basin", "date_start", "date_end", "date_center", "lat_025", "lon_025"],
                        as_index=False
                    )
                    .agg(
                        chlor_a_mean=("chlor_a", "mean"),
                        chlor_a_median=("chlor_a", "median"),
                        chlor_a_std=("chlor_a", "std"),
                        chlor_a_min=("chlor_a", "min"),
                        chlor_a_max=("chlor_a", "max"),
                        valid_pixel_count=("chlor_a", "count")
                    )
                )

                frames.append(df_025)

                print(basin_name, "rows:", len(df_025))

            except Exception as e:
                print("Region failed:", basin_name)
                print("Reason:", e)

        ds.close()

    except Exception as e:
        print("Failed to open OPeNDAP file:")
        print(url)
        print("Reason:", e)
        return pd.DataFrame()

    if len(frames) == 0:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

In [3]:
test_start = datetime(2020, 1, 1)
test_end = datetime(2020, 1, 8)

df_test = process_one_opendap_file(test_start, test_end)

print("Test shape:", df_test.shape)
df_test.head()


Processing: AQUA_MODIS.20200101_20200108.L3m.8D.CHL.chlor_a.4km.nc
Bay_of_Bengal rows: 3757
Arabian_Sea rows: 3744
Test shape: (7501, 12)


,basin,date_start,date_end,date_center,lat_025,lon_025,chlor_a_mean,chlor_a_median,chlor_a_std,chlor_a_min,chlor_a_max,valid_pixel_count
0,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,80.125,0.258539,0.264737,0.042121,0.172192,0.349827,36
1,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,80.375,0.176654,0.175468,0.015453,0.128524,0.226184,36
2,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,80.625,0.166483,0.175305,0.022700,0.118447,0.199029,36
3,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,80.875,0.154920,0.152771,0.023633,0.106967,0.200826,36
4,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,81.125,0.144689,0.142248,0.024714,0.108827,0.227319,35


In [4]:
all_frames = []
failed_periods = []

for year in range(START_YEAR, END_YEAR + 1):
    periods = generate_8day_periods(year)

    for period_start, period_end in tqdm(periods, desc=f"Year {year}"):
        df_temp = process_one_opendap_file(period_start, period_end)

        if df_temp.empty:
            failed_periods.append((period_start, period_end))
        else:
            all_frames.append(df_temp)

if len(all_frames) == 0:
    raise ValueError("No MODIS chlorophyll data processed. Check OPeNDAP access.")

modis_chl_025 = pd.concat(all_frames, ignore_index=True)

print("Final MODIS chlorophyll shape:", modis_chl_025.shape)
print("Failed periods:", len(failed_periods))

modis_chl_025.head()

Year 2020:   0%|          | 0/46 [00:00<?, ?it/s]


Processing: AQUA_MODIS.20200101_20200108.L3m.8D.CHL.chlor_a.4km.nc


Year 2020:   2%|▏         | 1/46 [00:01<00:47,  1.07s/it]

Bay_of_Bengal rows: 3757
Arabian_Sea rows: 3744

Processing: AQUA_MODIS.20200109_20200116.L3m.8D.CHL.chlor_a.4km.nc
Bay_of_Bengal rows: 3822


Year 2020:   4%|▍         | 2/46 [00:02<00:51,  1.18s/it]

...

Year 2024:  98%|█████████▊| 45/46 [00:52<00:01,  1.16s/it]

Bay_of_Bengal rows: 3188
Arabian_Sea rows: 3388

Processing: AQUA_MODIS.20241226_20241231.L3m.8D.CHL.chlor_a.4km.nc
Bay_of_Bengal rows: 3084


Year 2024: 100%|██████████| 46/46 [00:54<00:00,  1.18s/it]

Arabian_Sea rows: 3006
Final MODIS chlorophyll shape: (1205694, 12)
Failed periods: 1


,basin,date_start,date_end,date_center,lat_025,lon_025,chlor_a_mean,chlor_a_median,chlor_a_std,chlor_a_min,chlor_a_max,valid_pixel_count
0,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,80.125,0.258539,0.264737,0.042121,0.172192,0.349827,36
1,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,80.375,0.176654,0.175468,0.015453,0.128524,0.226184,36
2,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,80.625,0.166483,0.175305,0.022700,0.118447,0.199029,36
3,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,80.875,0.154920,0.152771,0.023633,0.106967,0.200826,36
4,Bay_of_Bengal,2020-01-01,2020-01-08,2020-01-04 12:00:00,5.125,81.125,0.144689,0.142248,0.024714,0.108827,0.227319,35


In [5]:
final_csv = OUTPUT_DIR / "modis_aqua_chlorophyll_8day_025deg_bob_as_2020_2024.csv"
final_parquet = OUTPUT_DIR / "modis_aqua_chlorophyll_8day_025deg_bob_as_2020_2024.parquet"

modis_chl_025.to_csv(final_csv, index=False)
modis_chl_025.to_parquet(final_parquet, index=False)

print("Saved CSV:", final_csv)
print("Saved Parquet:", final_parquet)

Saved CSV: modis_chlorophyll_data/processed/modis_aqua_chlorophyll_8day_025deg_bob_as_2020_2024.csv
Saved Parquet: modis_chlorophyll_data/processed/modis_aqua_chlorophyll_8day_025deg_bob_as_2020_2024.parquet


In [6]:
summary = (
    modis_chl_025.groupby(["basin", "date_start", "date_end", "date_center"], as_index=False)
    .agg(
        chlor_a_mean=("chlor_a_mean", "mean"),
        chlor_a_median=("chlor_a_median", "median"),
        chlor_a_std=("chlor_a_mean", "std"),
        chlor_a_min=("chlor_a_min", "min"),
        chlor_a_max=("chlor_a_max", "max"),
        valid_grid_cells=("valid_pixel_count", "count"),
        total_valid_pixels=("valid_pixel_count", "sum")
    )
)

summary_csv = OUTPUT_DIR / "modis_aqua_chlorophyll_8day_basin_summary_2020_2024.csv"
summary_excel = OUTPUT_DIR / "modis_aqua_chlorophyll_8day_basin_summary_2020_2024.xlsx"
summary_parquet = OUTPUT_DIR / "modis_aqua_chlorophyll_8day_basin_summary_2020_2024.parquet"

summary.to_csv(summary_csv, index=False)
summary.to_excel(summary_excel, index=False)
summary.to_parquet(summary_parquet, index=False)

print("Saved summary Excel:", summary_excel)
summary.head()

Saved summary Excel: modis_chlorophyll_data/processed/modis_aqua_chlorophyll_8day_basin_summary_2020_2024.xlsx


,basin,date_start,date_end,date_center,chlor_a_mean,chlor_a_median,chlor_a_std,chlor_a_min,chlor_a_max,valid_grid_cells,total_valid_pixels
0,Arabian_Sea,2020-01-01,2020-01-08,2020-01-04 12:00:00,0.415299,0.218771,1.013869,0.037991,53.132027,3744,126588
1,Arabian_Sea,2020-01-09,2020-01-16,2020-01-12 12:00:00,0.423762,0.229910,0.867328,0.033321,24.580702,3728,122933
2,Arabian_Sea,2020-01-17,2020-01-24,2020-01-20 12:00:00,0.563309,0.268910,1.350251,0.031221,39.372906,3696,120835
3,Arabian_Sea,2020-01-25,2020-02-01,2020-01-28 12:00:00,0.626203,0.274346,2.278628,0.059041,75.168282,3516,110424
4,Arabian_Sea,2020-02-02,2020-02-09,2020-02-05 12:00:00,0.623468,0.225414,1.969515,0.034876,82.651932,3778,122667


In [7]:
import shutil
from google.colab import files

zip_name = "modis_aqua_chlorophyll_2020_2024_processed"

shutil.make_archive(zip_name, "zip", "modis_chlorophyll_data")

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>